##Natural Language Processing: Incremental Capstone - Aamir Mohammed

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_csv("bike_rental_reviews.csv")
df.head()

,review_text,sentiment
0,"The entire process was easy, and the availabil...",positive
1,Standard rental process. The mobile app was ac...,neutral
2,One of the best bike rentals I’ve had. The mob...,positive
3,One of the best bike rentals I’ve had. The cus...,positive
4,Not worth the money. The seat comfort was a ma...,negative


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   review_text  50000 non-null  object
 1   sentiment    50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [ ]:
df.duplicated().sum()

np.int64(49700)

In [ ]:
df['sentiment'].value_counts()

,count
sentiment,
negative,16840
positive,16777
neutral,16383


In [ ]:
df['word_count'] = df['review_text'].str.split().str.len()
df[['review_text','word_count']].head()

,review_text,word_count
0,"The entire process was easy, and the availabil...",11
1,Standard rental process. The mobile app was ac...,8
2,One of the best bike rentals I’ve had. The mob...,15
3,One of the best bike rentals I’ve had. The cus...,15
4,Not worth the money. The seat comfort was a ma...,11


In [ ]:
df['word_count'].describe()

,word_count
count,50000.000000
mean,9.401700
std,1.895914
min,5.000000
25%,8.000000
50%,9.000000
75%,11.000000
max,15.000000


In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

stop_words = set(stopwords.words('english'))
negative_words = {"no", "not", "nor", "never"}
stop_words = stop_words - negative_words
lemmatizer = WordNetLemmatizer()


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


In [ ]:
from nltk.tag import pos_tag
def preprocess_text(text):
  text = text.lower()

  text = re.sub(r"[^a-z\s]", " ", text)

  words = word_tokenize(text)

  words = [word for word in words if word not in stop_words]

  lemmatized_tok = [lemmatizer.lemmatize(word) for word in words]
  return " ".join(lemmatized_tok)

In [ ]:
df['Clean Text'] = df['review_text'].apply(preprocess_text)
df.head()

,review_text,sentiment,word_count,Clean Text
0,"The entire process was easy, and the availabil...",positive,11,entire process easy availability high quality
1,Standard rental process. The mobile app was ac...,neutral,8,standard rental process mobile app acceptable
2,One of the best bike rentals I’ve had. The mob...,positive,15,one best bike rental mobile app made even better
3,One of the best bike rentals I’ve had. The cus...,positive,15,one best bike rental customer service made eve...
4,Not worth the money. The seat comfort was a ma...,negative,11,not worth money seat comfort major letdown


---
## Logistic Regression Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X = df['Clean Text']
target = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    target,
    test_size=0.2,
    random_state=100,
    stratify=target
)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)
X_train_matrix = tf.fit_transform(X_train)
X_test_matrix = tf.transform(X_test)

In [ ]:
mdl = LogisticRegression(max_iter=5000)
mdl.fit(X_train_matrix, y_train)

LogisticRegression(max_iter=5000)

In [ ]:
y_pred = mdl.predict(X_test_matrix)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred))
print('\n', classification_report(y_test, y_pred))
print('\n', confusion_matrix(y_test, y_pred))

Logistic Regression Accuracy: 1.0

               precision    recall  f1-score   support

    negative       1.00      1.00      1.00      3368
     neutral       1.00      1.00      1.00      3277
    positive       1.00      1.00      1.00      3355

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000


 [[3368    0    0]
 [   0 3277    0]
 [   0    0 3355]]


---
## Naive Bayes Model

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb_mdl = MultinomialNB()
nb_mdl.fit(X_train_matrix, y_train)

MultinomialNB()

In [ ]:
y_pred_nb = nb_mdl.predict(X_test_matrix)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print('\n', classification_report(y_test, y_pred_nb))
print('\n', confusion_matrix(y_test, y_pred_nb))

Naive Bayes Accuracy: 1.0

               precision    recall  f1-score   support

    negative       1.00      1.00      1.00      3368
     neutral       1.00      1.00      1.00      3277
    positive       1.00      1.00      1.00      3355

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000


 [[3368    0    0]
 [   0 3277    0]
 [   0    0 3355]]


---
## Deep Learning - LSTM

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

from sklearn.preprocessing import LabelEncoder

In [ ]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)


In [ ]:
vocab_size = 10000
embedding_dim = 100
max_length = 100

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_test_sequences = tokenizer.texts_to_sequences(X_test)

X_train_padded = pad_sequences(X_train_sequences, maxlen=max_length, padding='post', truncating='post')
X_test_padded = pad_sequences(X_test_sequences, maxlen=max_length, padding='post', truncating='post')

print(X_train_padded.shape, X_test_padded.shape)
print(len(y_train_encoded), len(y_test_encoded))

(40000, 100) (10000, 100)
40000 10000


In [ ]:
model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_length),
    LSTM(64, return_sequences=False),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train_padded,
    y_train_encoded,
    validation_split=0.2,
    epochs=5,
    batch_size=256,
    verbose=1
)

Epoch 1/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 20s 33ms/step - accuracy: 0.3363 - loss: 1.0990 - val_accuracy: 0.3290 - val_loss: 1.0988
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 8s 29ms/step - accuracy: 0.3323 - loss: 1.0987 - val_accuracy: 0.3309 - val_loss: 1.0986
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.3360 - loss: 1.0986 - val_accuracy: 0.3309 - val_loss: 1.0986
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.3322 - loss: 1.0987 - val_accuracy: 0.3309 - val_loss: 1.0986
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 37ms/step - accuracy: 0.3421 - loss: 1.0986 - val_accuracy: 0.3309 - val_loss: 1.0986


In [ ]:
# classification and confusion matrix

y_pred_pr = model.predict(X_test_padded)
y_pred_lstm = y_pred_pr.argmax(axis=1)

print("LSTM Accuracy:", accuracy_score(y_test_encoded, y_pred_lstm))
print('\nClassification Report:\n', classification_report(y_test_encoded, y_pred_lstm, target_names=le.classes_))
print('\nConfusion Matrix\n', confusion_matrix(y_test_encoded, y_pred_lstm))



313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step
LSTM Accuracy: 0.3355

Classification Report:
               precision    recall  f1-score   support

    negative       0.00      0.00      0.00      3368
     neutral       0.00      0.00      0.00      3277
    positive       0.34      1.00      0.50      3355

    accuracy                           0.34     10000
   macro avg       0.11      0.33      0.17     10000
weighted avg       0.11      0.34      0.17     10000


Confusion Matrix
 [[   0    0 3368]
 [   0    0 3277]
 [   0    0 3355]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


---
### Transformer Model

In [ ]:
!pip install transformers

In [ ]:
from transformers import pipeline, AutoTokenizer, TFBertForSequenceClassification, DataCollatorWithPadding

In [ ]:
X = df['Clean Text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=100,
    stratify=y
)

In [ ]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

In [ ]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
transformer_model = TFBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    from_pt=True
    )



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
#tokenize text

train_encodings = tokenizer(
    list(X_train),
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors='tf'
)

test_encodings = tokenizer(
    list(X_test),
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors='tf'
)

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    y_train_encoded
)).shuffle(1000).batch(32)

test_dataset = tf.data.Dataset.from_tensor_slices((
    dict(test_encodings),
    y_test_encoded
)).batch(32)

In [ ]:
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy

opt = Adam(learning_rate=5e-5)

transformer_model.compile(
    optimizer='Adam',
    loss=SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [ ]:
history_ = transformer_model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=3
)

Epoch 1/3
1250/1250 [==============================] - 266s 130ms/step - loss: 1.1426 - accuracy: 0.3322 - val_loss: 1.1006 - val_accuracy: 0.3355
Epoch 2/3
1250/1250 [==============================] - 143s 114ms/step - loss: 1.1307 - accuracy: 0.3311 - val_loss: 1.1132 - val_accuracy: 0.3277
Epoch 3/3
1250/1250 [==============================] - 142s 114ms/step - loss: 1.1145 - accuracy: 0.3322 - val_loss: 1.1024 - val_accuracy: 0.3368


In [ ]:
y_pred_tr = transformer_model.predict(test_dataset)
y_pred_tr = np.argmax(y_pred_tr.logits, axis=1)

print("BERT Accuracy:", accuracy_score(y_test_encoded, y_pred_tr))
print('\nClassification Report:\n', classification_report(y_test_encoded, y_pred_tr, target_names=le.classes_))
print('\nConfusion Matrix\n', confusion_matrix(y_test_encoded, y_pred_tr))

313/313 [==============================] - 27s 86ms/step
BERT Accuracy: 0.3368

Classification Report:
               precision    recall  f1-score   support

    negative       0.34      1.00      0.50      3368
     neutral       0.00      0.00      0.00      3277
    positive       0.00      0.00      0.00      3355

    accuracy                           0.34     10000
   macro avg       0.11      0.33      0.17     10000
weighted avg       0.11      0.34      0.17     10000


Confusion Matrix
 [[3368    0    0]
 [3277    0    0]
 [3355    0    0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Final Findings
### Traditional Models:
- Both the Logistic Regression model and Naive Bayes model achieved 100% accuracy in classification of the reviews as positive, negative, or neutral
- F-1 scores, recall were all perfect across all sentiment classes
- And the Confusion matrix also showed no faulty classifications in either model
- THese findings showed to me that the dataset and cleaning the data allowed for it to be highly easy to seperate and classify using TF-IDF features.
- These traditional ML approaches were fast, stable, and very effective in classification compared to Deep Learning and Transformer Models

### LSTM Deep Learning and BERT Transformer Model
- Both LSTM and BERT achieved around ~33% accuracy in classification of the sentiment analysis.
- The models seemingly chose 1 sentiment class to classify all reviews into 1 dominant class, leading to bad recall, precision scores for the other sentiment classes
- Training them seemed to be innefective as well for some reason across epochs, indicating weak learning from these models on this specific dataset
- Some causes for this I thought could have been due to:
  - The dataset being simple to seperate and classify, which led to the highly complex models adding unnecessary complexity
  - limitazion of truncation or tokenization

- Changed few things to see if I can create better models but ultimately I couldn't find a good way to make the models better without having to clean the text properly

### Key Takeaway
- The Traditional ML Models were outperforming the highly complex, and more computationaly expensive models, showing they were less effective and harder to optimize in classifying sentiment of the reviews properly